In [1]:
import os
import pandas as pd
import numpy as np

def analyze_customer_leak_with_type(file_path, folder_type, time_col='Hourly', consumption_col='Consumption m3'):
    """
    Processes any customer dataset. Uses an industry-standard 8-week rolling baseline 
    to evaluate the most recent 4-week window for slow leaks and sudden bursts.
    """
    try:
        # --- 1. Load, Sort, and Clean Data ---
        # Kept header=2 to perfectly read starting from Row 3 of your Excel sheets
        df = pd.read_excel(file_path, header=2) 
            
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.sort_values(time_col).reset_index(drop=True)
        
        # Extract Time Features
        df['hour'] = df[time_col].dt.hour
        df['day_of_week'] = df[time_col].dt.dayofweek
        
        # ROLLING TIMELINE WINDOWS (Anchored to the max date to handle infinite data quantity)
        max_date = df[time_col].max()
        four_weeks_ago = max_date - pd.Timedelta(weeks=4)
        eight_weeks_prior = four_weeks_ago - pd.Timedelta(weeks=8)
        
        # Classify periods dynamically based on lookbacks
        df['period'] = 'Ignore' # Anything older than 12 weeks total is safely ignored to maintain performance
        df.loc[(df[time_col] >= eight_weeks_prior) & (df[time_col] < four_weeks_ago), 'period'] = 'Historical_Baseline'
        df.loc[df[time_col] >= four_weeks_ago, 'period'] = 'Recent_Evaluation'
        
        # --- 2. Build Baseline Profiles (FIXED: Using clean past data to avoid pollution) ---
        baseline_data = df[df['period'] == 'Historical_Baseline']
        
        # Data Gap Fallback: If 8-week window is entirely empty due to a missing data chunk
        if baseline_data.empty:
            baseline_data = df[df[time_col] < four_weeks_ago] # Use any available history before the test window
        if baseline_data.empty:
            baseline_data = df # Last resort fallback if dataset is incredibly short
            
        baseline_profile = baseline_data.groupby(['day_of_week', 'hour'])[consumption_col].median().reset_index()
        baseline_profile.rename(columns={consumption_col: 'baseline_median'}, inplace=True)
        
        baseline_std = baseline_data.groupby(['day_of_week', 'hour'])[consumption_col].std().reset_index()
        baseline_std.rename(columns={consumption_col: 'baseline_std'}, inplace=True)
        
        # Merge clean historical profiles back onto the main dataframe
        df = pd.merge(df, baseline_profile, on=['day_of_week', 'hour'], how='left')
        df = pd.merge(df, baseline_std, on=['day_of_week', 'hour'], how='left')
        
        # --- 3. Compute Structural Deviations (FIXED: Stabilized math) ---
        df['abs_deviation'] = df[consumption_col] - df['baseline_median']
        
        # Prevent division-by-zero or near-infinite Z-scores for users with clean zero-consumption nights
        global_median = df[consumption_col].median()
        std_safety_floor = max(0.005, global_median * 0.1) # 10% of median usage or 5 liters minimum
        
        df['z_score'] = df['abs_deviation'] / (df['baseline_std'] + std_safety_floor)
        df[['abs_deviation', 'z_score']] = df[['abs_deviation', 'z_score']].fillna(0)
        
        # --- 4. DYNAMIC FOLDER-BASED THRESHOLD ASSIGNMENT ---
        if any(kw in folder_type for kw in ["Residential", "Villa", "Flat"]):
            night_hours = [1, 2, 3, 4]
            night_drift_threshold = 0.025     
            z_threshold = 3.0                 
            consecutive_hours = 2             
            
        elif "Government" in folder_type:
            night_hours = [0, 1, 5, 6]        
            night_drift_threshold = 0.500     
            z_threshold = 4.5                 
            consecutive_hours = 5             
            
        elif "Industrial" in folder_type:
            night_hours = [2, 3, 4]              
            night_drift_threshold = 1.000     
            z_threshold = 5.0                 
            consecutive_hours = 6             
            
        else: # "Commercial" or "Hotel"
            night_hours = [1, 2, 3, 4]
            night_drift_threshold = 0.150     
            z_threshold = 3.5
            consecutive_hours = 4

        # --- 5. Evaluate Automation Logic ---
        leak_detected = False
        leak_reasons = []
        
        # Pull night values strictly from the clean historical baseline window vs. the last 7 days
        hist_night = df[(df['period'] == 'Historical_Baseline') & (df['hour'].isin(night_hours))]
        recent_week_night = df[(df[time_col] >= (max_date - pd.Timedelta(days=7))) & (df['hour'].isin(night_hours))]
        
        historical_min_flow = hist_night[consumption_col].quantile(0.05) if not hist_night.empty else 0
        recent_min_flow = recent_week_night[consumption_col].min() if not recent_week_night.empty else 0
        
        # Unified Night Flow Check
        if recent_min_flow > (historical_min_flow + night_drift_threshold):
            if recent_min_flow > 0.03: 
                leak_detected = True
                leak_reasons.append(f"Slow Leak: Constant minimum night baseline has lifted from {historical_min_flow:.3f} m3 up to {recent_min_flow:.3f} m3.")
        
        # Daytime Burst Evaluation
        df['is_anomaly'] = (df['period'] == 'Recent_Evaluation') & (df['z_score'] > z_threshold)
        consecutive_anomalies = df['is_anomaly'].astype(int).rolling(window=consecutive_hours).sum()
        
        if (consecutive_anomalies >= consecutive_hours).any():
            leak_detected = True
            # FIXED: Target the exact row indices where the anomaly happened to grab the correct peak Z-Score
            anomalous_indices = consecutive_anomalies[consecutive_anomalies >= consecutive_hours].index
            max_z = df.loc[anomalous_indices, 'z_score'].max()
            leak_reasons.append(f"Sudden Burst: High usage spike sustained for {consecutive_hours}+ hours (Peak Local Z-Score: {max_z:.1f}).")
            
        # --- 6. Output Summary Dashboard ---
        print("="*85)
        print(f"AUTOMATED LEAK AUDIT REPORT: {os.path.basename(file_path)}")
        print(f"Subfolder Profile Read: {folder_type.upper()}")
        print("="*85)
        
        if leak_detected:
            print("🔴 STATUS: LEAK SUSPECTED")
            for reason in leak_reasons:
                print(f"  - {reason}")
        else:
            print("🟢 STATUS: NORMAL (NO LEAK DETECTED)")
        print("="*85 + "\n")
        
        return {
            "Filename": os.path.basename(file_path),
            "Profile_Folder": folder_type,
            "Leak_Suspected": "YES" if leak_detected else "NO",
            "Details": "; ".join(leak_reasons) if leak_reasons else "Normal usage patterns.",
            "Historical_Night_Min": round(historical_min_flow, 4),
            "Recent_Night_Min": round(recent_min_flow, 4)
        }
        
    except Exception as e:
        print(f"❌ Failure for file {file_path}: {str(e)}\n")
        return {
            "Filename": os.path.basename(file_path),
            "Profile_Folder": folder_type,
            "Leak_Suspected": "ERROR",
            "Details": f"Processing failure: {str(e)}",
            "Historical_Night_Min": 0,
            "Recent_Night_Min": 0
        }


In [ ]:
def run_portfolio_leak_audit(base_folder="customers"):
    """
    Crawls through the 'base_folder', finds all Excel files inside category 
    subfolders (e.g., 'Villa (Residential)', 'Commercial'), and scores them.
    """
    results_list = []
    
    if not os.path.exists(base_folder):
        print(f"❌ Directory path error: Base folder '{base_folder}' does not exist.")
        return results_list
        
    print(f"🚀 Initializing Portfolio Scanning Pipeline on directory: '{base_folder}'...\n")
    
    # Walk through folders, subfolders, and files
    for root, dirs, files in os.walk(base_folder):
        for file in files:
            # Targets both standard excel formats (.xlsx, .xls)
            if file.endswith(('.xlsx', '.xls')) and not file.startswith('~$'):
                full_file_path = os.path.join(root, file)
                
                # Capture the name of the immediate parent folder to use as the dynamic profile type
                folder_category = os.path.basename(root)
                
                # Execute audit logic for the file
                audit_summary = analyze_customer_leak_with_type(
                    file_path=full_file_path, 
                    folder_type=folder_category
                )
                results_list.append(audit_summary)
                
    return results_list

# =====================================================================
# PIPELINE EXECUTION ENGINE 
# =====================================================================

# 1. Run the cleaner pipeline on your 'customers' folder tree
all_audit_results = run_portfolio_leak_audit(base_folder="customers")

# 2. Automatically transform results array into a structured DataFrame if files were found
if all_audit_results:
    summary_df = pd.DataFrame(all_audit_results)

    # 3. Export directly to a master tracking CSV sheet in your project workspace
    output_file = "portfolio_leakage_audit_summary.csv"
    summary_df.to_csv(output_file, index=False)

    print(f"💾 SUCCESS: Master audit log ledger exported to: '{output_file}'!")
    print(summary_df.head())
else:
    print("⚠️ Pipeline finished: No Excel data records found or processed inside target directory tree.")